# 1116 52일차

## 1. 결측치

값이 들어 있지 않은 칸 (`pd01_imputation1_interpolate`, `pd02_imputation2_pandas`)

사이킷런·케라스 모델은 결측치가 있으면 학습을 시작하지 못하므로, 지울지 채울지를 먼저 정해야 함

1. 삭제 : `dropna()`로 결측치가 있는 행이나 열을 통째로 버림
2. 임의의 값 채우기 : `fillna()`에 0·평균·중위값·최빈값·정해 둔 값을 넣음
3. 앞뒤 값 채우기 : `ffill()`은 앞의 값, `bfill()`은 뒤의 값
4. 보간 : 앞뒤 값 사이를 이어 중간값을 만듦
5. 모델 : 결측 열을 y로 두고 나머지 열로 예측해 채움
6. 트리·부스팅 계열 : 결측치와 이상치의 영향을 덜 받아 채우지 않고 그대로 넣기도 함

- `df.isnull()`은 칸마다 True/False를 돌려주고, `.sum()`을 붙이면 열별 결측치 개수가 나옴
- `df.info()`의 `Non-Null Count`로도 몇 개가 비었는지 보임
- **`dropna()`의 기본은 행 삭제. `axis=1`을 주면 열을 지움**
- `fillna(data.mean())`처럼 Series를 넣으면 열마다 그 열의 값으로 채움
- 평균은 이상치에 끌려가므로 중위값을 쓰기도 함
- 열마다 방법을 달리하려면 `data['x1'] = data['x1'].fillna(means)`처럼 열을 따로 지정
- 채우는 함수는 원본을 바꾸지 않아 결과를 다시 대입해야 함

## 2. interpolate

알려진 값 사이에 새 값을 만들어 끼워 넣는 것 (`pd01_imputation1_interpolate`)

순서가 있는 데이터는 앞뒤 값을 이어서 채우는 쪽이 열 전체의 평균으로 채우는 것보다 실제 값에 가까움

1. `ts.interpolate()` : 기본은 linear. 앞뒤 값을 직선으로 이어 등간격으로 채움
2. `pd.to_datetime(dates)` : 날짜 문자열 목록을 `DatetimeIndex`로 바꿈. 값이 아니라 인덱스로 쓰는 것

- **중간이 비면 앞의 값과 뒤의 값을 직선으로 이어 채움**
- 연속으로 비어 있어도 양쪽 값만 있으면 한 번에 채워짐
- 마지막이 비면 이을 뒤 값이 없어 앞의 값이 그대로 들어감(`ffill`과 같은 결과)
- 첫 번째가 비면 앞 값이 없어 NaN으로 남음
- 인덱스가 날짜여도 기본 linear는 간격을 보지 않고 순서만 봄
- 결과는 새 Series라 `ts = ts.interpolate()`로 다시 대입해야 함

## 3. Imputer

사이킷런에서 결측치를 채우는 변환기 (`pd03_imputation3_imputer1`, `pd03_imputation3_imputer2`, `pd04_imputation4_KNNImputer`, `pd05_imputation5_IterativeImputer`)

스케일러와 같은 `fit_transform` 구조라, train에서 구한 값을 test에도 그대로 적용할 수 있음

1. `SimpleImputer()` : 열마다 통계값 하나로 채움. 기본은 평균
2. `strategy` : `'mean'`·`'median'`·`'most_frequent'`(최빈값)·`'constant'`
3. `SimpleImputer(strategy='constant', fill_value=777)` : 정해 둔 값 하나로 채움
4. `SimpleImputer(strategy=lambda x: np.min(x))` : 함수를 넘기면 열마다 그 함수의 결과로 채움
5. `KNNImputer(n_neighbors=2)` : 나머지 열이 비슷한 행 k개를 찾아 그 값들의 평균으로 채움
6. `IterativeImputer(estimator=xgb, max_iter=10, tol=0.001)` : 결측 열을 나머지 열로 예측하는 것을 열마다 돌려가며 반복

- **`fit_transform`은 넘파이 배열을 돌려줘서 열 이름이 사라짐**
- `SimpleImputer`는 그 열만 보고, `KNNImputer`·`IterativeImputer`는 다른 열의 값까지 보고 채움
- `n_neighbors`가 작을수록 가까운 행만 보고, 클수록 열 평균에 가까워짐
- `estimator`에 쓸 회귀 모델을 넣음. 생략하면 `BayesianRidge`
- 채운 값의 변화가 `tol`보다 작아지면 `max_iter`를 채우지 않고 멈춤
- `imputer.n_iter_`로 실제 반복 횟수를 확인함

#### cf) 실험 단계 기능

- `IterativeImputer`는 아직 실험 단계라 `from sklearn.experimental import enable_iterative_imputer`를 먼저 적어야 import됨
- 이 줄은 코드에서 쓰이지 않는 것처럼 보여도 지우면 `IterativeImputer`를 못 불러옴

## 4. loc와 iloc

행을 고르는 두 가지 방법 (`pd06_loc_iloc1`, `pd06_loc_iloc2_range`)

대괄호 하나(`df[...]`)는 열 이름만 받기 때문에, 행을 뽑으려면 따로 필요함

1. `loc` : 인덱스, 즉 행 이름으로 고름
2. `iloc` : 행 번호로 고름. integer location으로 외움
3. `df['종목명']` : 열 하나를 Series로 꺼냄
4. `df.loc['045', '종가']` : 콤마 왼쪽이 행, 오른쪽이 열
5. `df.iloc[3:5, 1]` : 여러 행을 고르는 슬라이싱도 콤마 왼쪽에 들어감

- **판다스는 열이 먼저. 대괄호를 쌓을 때도 `df['종목명']['045']` 순서로 들어감**
- `df[3, 1]`·`df[3][1]`·`df['045']`처럼 대괄호 하나에 행을 넣으면 `KeyError`
- `iloc`에 이름을, `loc`에 번호를 넣으면 에러
- 콤마 안에서는 `iloc`은 번호끼리, `loc`은 이름끼리만 씀
- `loc` 슬라이싱은 끝 이름을 포함하고, `iloc`은 끝 번호를 포함하지 않음
- 인덱스가 `'031'`처럼 숫자 모양 문자열이면 `loc[3]`은 그런 이름이 없어 에러

#### cf) 대괄호 나눠 쓰기

- 행을 먼저 꺼내면 결과가 Series라, 그다음은 `iloc[2]`로도 `loc['종가']`로도 꺼내짐
- `df.iloc[3]['종가']`는 되지만 `df.iloc[3][2]`는 pandas 3.0에서 에러(2버전대에서는 됐음)
- 슬라이싱한 뒤에는 행이 여럿이라 `df.iloc[3:5].loc['시가']`처럼 열 이름을 바로 붙일 수 없음
- **한 행 한 열을 고를 때는 대괄호를 쌓지 않고 `df.loc[행, 열]`로 한 번에 씀**